In [3]:
import os, math, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.utils import to_undirected

In [4]:
import scipy.sparse as sp
from sklearn.preprocessing import normalize
from sklearn.decomposition import TruncatedSVD

In [5]:
def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [6]:
def load_graph_from_folder(folder: str) -> Data:
    edges = np.loadtxt(os.path.join(folder, "edges.txt"), dtype=np.int64)
    X = np.loadtxt(os.path.join(folder, "features.csv"), delimiter=",", dtype=np.float32)
    y = np.loadtxt(os.path.join(folder, "labels.csv"), dtype=np.int64)

    edge_index = torch.tensor(edges.T, dtype=torch.long)
    edge_index = to_undirected(edge_index)

    data = Data(
        x=torch.tensor(X, dtype=torch.float),
        y=torch.tensor(y, dtype=torch.long),
        edge_index=edge_index
    )
    return data

In [7]:
def read_idx(path: str, add_one: bool = False) -> torch.Tensor:
    idx = np.loadtxt(path, dtype=np.int64)
    if idx.ndim == 0:
        idx = np.array([int(idx)], dtype=np.int64)
    if add_one:
        idx = idx + 1
    return torch.tensor(idx, dtype=torch.long)

In [8]:
def apply_split_masks(data: Data, train_idx, val_idx, test_idx):
    n = data.num_nodes
    train_mask = torch.zeros(n, dtype=torch.bool); train_mask[train_idx] = True
    val_mask = torch.zeros(n, dtype=torch.bool); val_mask[val_idx] = True
    test_mask = torch.zeros(n, dtype=torch.bool); test_mask[test_idx] = True
    data.train_mask = train_mask
    data.val_mask = val_mask
    data.test_mask = test_mask
    return data

In [9]:
class RunPreprocessor:
    def __init__(self, use_tfidf=True, l2_rows=True, pca_dim=256):
        self.use_tfidf = use_tfidf
        self.l2_rows = l2_rows
        self.pca_dim = int(pca_dim)
        self.idf = None
        self.svd = None
    def fit(self, X_all_dense: np.ndarray, train_mask: np.ndarray):
        X_all = sp.csr_matrix(X_all_dense)
        X_tr = X_all[train_mask]
        if self.use_tfidf:
            df = np.asarray((X_tr > 0).sum(axis=0)).reshape(-1)
            Ntr = X_tr.shape[0]
            self.idf = np.log((Ntr + 1.0) / (df + 1.0)) + 1.0
        else:
            self.idf = None
        Xw = X_tr
        if self.idf is not None:
            Xw = Xw.multiply(self.idf)
        if self.l2_rows:
            Xw = normalize(Xw, norm="l2", axis=1)
        max_rank = max(1, min(self.pca_dim, Xw.shape[1] - 1, Xw.shape[0] - 1))
        self.svd = TruncatedSVD(n_components=max_rank, random_state=0)
        self.svd.fit(Xw)
        return self
    def transform(self, X_all_dense: np.ndarray) -> np.ndarray:
        X_all = sp.csr_matrix(X_all_dense)
        Xw = X_all
        if self.idf is not None:
            Xw = Xw.multiply(self.idf)
        if self.l2_rows:
            Xw = normalize(Xw, norm="l2", axis=1)
        Xr = self.svd.transform(Xw) 
        return Xr.astype(np.float32)

In [10]:
def preprocess_features_per_run(data: Data, use_raw: bool,
     use_tfidf=True, l2_rows=True, pca_dim=256):
     X_dense = data.x.detach().cpu().numpy()
     if use_raw:
         return data # no change
     pre = RunPreprocessor(use_tfidf=use_tfidf, l2_rows=l2_rows, pca_dim=pca_dim)
     pre.fit(X_dense, data.train_mask.detach().cpu().numpy())
     Xp = pre.transform(X_dense)
     data.x = torch.tensor(Xp, dtype=torch.float)
     return data